In [1]:
import os

# adjust <dataset-folder-name> to whatever Kaggle assigned as the top-level input folder
BASE = "/kaggle/input/datasets/apollo2506/eurosat-dataset/EuroSAT"

# confirm this resolves correctly before proceeding
print(os.listdir(BASE))

['SeaLake', 'Highway', 'River', 'Pasture', 'Industrial', 'Residential', 'PermanentCrop', 'validation.csv', 'AnnualCrop', 'train.csv', 'test.csv', 'label_map.json', 'Forest', 'HerbaceousVegetation']


In [2]:
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset

class EuroSATCSVDataset(Dataset):
    def __init__(self, csv_path, root_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row['Filename'])
        image = Image.open(img_path).convert('RGB')
        label = int(row['Label'])
        if self.transform:
            image = self.transform(image)
        return image, label

from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = EuroSATCSVDataset(os.path.join(BASE, "train.csv"), BASE, transform=transform)
val_ds = EuroSATCSVDataset(os.path.join(BASE, "validation.csv"), BASE, transform=transform)

print("Train images:", len(train_ds))
print("Val images:", len(val_ds))

# the real test -- does the file path actually resolve and load?
img, label = train_ds[0]
print("Sample image shape:", img.shape, "| label:", label)

Train images: 18900
Val images: 5400
Sample image shape: torch.Size([3, 64, 64]) | label: 0


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
import os

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,
                           num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False,
                         num_workers=4, pin_memory=True, persistent_workers=True)

CLASS_NAMES = ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial',
               'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']

model_cls = models.resnet18(weights='IMAGENET1K_V1')
model_cls.fc = nn.Linear(model_cls.fc.in_features, len(CLASS_NAMES))
model_cls = model_cls.cuda()

optimizer = torch.optim.AdamW(model_cls.parameters(), lr=3e-5, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler()

os.makedirs('/kaggle/working/eurosat_checkpoints', exist_ok=True)
best_val_acc = 0.0

for epoch in range(20):
    model_cls.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.cuda(non_blocking=True), labels.cuda(non_blocking=True)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model_cls(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    model_cls.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.cuda(), labels.cuda()
            with torch.cuda.amp.autocast():
                outputs = model_cls(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    val_acc = 100 * correct / total
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}, Val Acc: {val_acc:.2f}%")

    torch.save(model_cls.state_dict(), '/kaggle/working/eurosat_checkpoints/last.pt')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model_cls.state_dict(), '/kaggle/working/eurosat_checkpoints/best.pt')

print(f"\nBest validation accuracy: {best_val_acc:.2f}%")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 179MB/s]
/tmp/ipykernel_23/3434949856.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_23/3434949856.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/3434949856.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1, Loss: 0.5772, Val Acc: 93.80%
Epoch 2, Loss: 0.1445, Val Acc: 95.39%
Epoch 3, Loss: 0.0817, Val Acc: 95.80%
Epoch 4, Loss: 0.0435, Val Acc: 96.07%
Epoch 5, Loss: 0.0318, Val Acc: 96.13%
Epoch 6, Loss: 0.0228, Val Acc: 96.35%
Epoch 7, Loss: 0.0174, Val Acc: 96.19%
Epoch 8, Loss: 0.0089, Val Acc: 96.11%
Epoch 9, Loss: 0.0104, Val Acc: 96.24%
Epoch 10, Loss: 0.0075, Val Acc: 96.30%
Epoch 11, Loss: 0.0091, Val Acc: 96.28%
Epoch 12, Loss: 0.0070, Val Acc: 96.20%
Epoch 13, Loss: 0.0081, Val Acc: 95.70%
Epoch 14, Loss: 0.0075, Val Acc: 96.33%
Epoch 15, Loss: 0.0056, Val Acc: 96.39%
Epoch 16, Loss: 0.0109, Val Acc: 95.96%
Epoch 17, Loss: 0.0072, Val Acc: 96.44%
Epoch 18, Loss: 0.0053, Val Acc: 96.39%
Epoch 19, Loss: 0.0035, Val Acc: 96.54%
Epoch 20, Loss: 0.0050, Val Acc: 96.20%

Best validation accuracy: 96.54%
